In [ ]:
install.packages("tictoc")


In [ ]:
install.packages("correlation")

In [ ]:
install.packages("missForest")

In [ ]:
install.packages("plyr")

In [ ]:
install.packages("doParallel")

In [ ]:
install.packages("parallel")

In [ ]:
install.packages("dplyr")

In [ ]:
install.packages("purrr")

In [ ]:
install.packages("UpSetR")


In [ ]:
# Install the package if not already installed
install.packages("TCGAretriever")

# Load the package
library(TCGAretriever)

In [ ]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("curatedTCGAData", dependencies = TRUE, force = TRUE)
BiocManager::install("RaggedExperiment", force = TRUE)
BiocManager::install("colData", force = TRUE)
BiocManager::install("SummarizedExperiment", force = TRUE)
BiocManager::install("MultiAssayExperiment", force = TRUE)

In [ ]:
# Load the library
library(curatedTCGAData)
library(RaggedExperiment)
library(UpSetR)
library(purrr)
library(dplyr)
library(SummarizedExperiment)
library(parallel)
library(doParallel)
library(plyr)
library(missForest)
library(correlation)
library(tictoc)
library(MultiAssayExperiment)

In [ ]:
curatedTCGAData(diseaseCode = c("LUAD", "LUSC"), version = "2.1.1")

In [ ]:
NSCLCdata <- curatedTCGAData(c("LUAD", "LUSC"), c("GISTIC_ThresholdedByGene","Mutation", "RNASeq2GeneNorm", "RPPAArray"),
                             dry.run = FALSE,
                             version = "2.1.1")

In [ ]:
dim(colData(NSCLCdata))

In [ ]:
table(NSCLCdata$histological_type)

In [ ]:
upsetSamples(NSCLCdata)

In [ ]:
## col_data for graph annotation
dfcolData <- data.frame(matrix(nrow = nrow(colData(NSCLCdata))))
for(i in 1:ncol(colData(NSCLCdata))){dfcolData <- cbind(dfcolData, NSCLCdata@colData@listData[i])}
dfcolData <- dfcolData[,-1]
rownames(dfcolData) <- NSCLCdata@colData@rownames
colnames(dfcolData) <- gsub("\\-|\\_", ".", colnames(dfcolData))
rownames(dfcolData) <- gsub("\\-|\\_", ".", rownames(dfcolData))
toBeRemoved <- which(data.frame(map(dfcolData, ~sum(is.na(.))))> nrow(dfcolData)*0.3)# just to reduce the dimension, not for imputing
dfcolData <- dfcolData[,-(toBeRemoved)]

# select patients with only ductal or lobular carcinoma
dfcolData <- dfcolData %>% filter(histological.type == "lung adenocarcinoma- not otherwise specified (nos)" | histological.type == "lung squamous cell carcinoma- not otherwise specified (nos)" | histological.type == "lung squamous cell carcinoma- not otherwise specified (nos)")

######
listAssays <- assays(NSCLCdata)


In [ ]:
# Assuming the code you provided successfully prints the gene symbols
gene_symbols <- NSCLCdata@ExperimentList@listData$'LUAD_GISTIC_ThresholdedByGene-20160128'@elementMetadata$'Gene.Symbol'

# Create the dataframe
gene_symbol_df <- data.frame(Gene.Symbol = gene_symbols)


head(gene_symbol_df)

In [ ]:
# Check the structure of NSCLCdata
str(NSCLCdata)


In [ ]:
# Assuming LUAD_GISTIC_ThresholdedByGene-20160128 is a data frame in your NSCLCdata
gistic_data <- NSCLCdata$LUAD_GISTIC_ThresholdedByGene_20160128

# Print the first few rows to inspect the data
head(gistic_data)

In [ ]:
print(listAssays[8])

In [ ]:
head(listAssays[8])

In [ ]:
# Assuming listAssays[1] is a data frame
data_frame <- listAssays[[5]]

# Print the contents using the head() function
head(data_frame)


In [ ]:
#######
## RNA-Seq
RNASeq <- listAssays[[7]]
colnames(RNASeq) <- gsub("(\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[0-9]+)$", "", colnames(RNASeq))
colnames(RNASeq) <- gsub("\\-|\\_", ".", colnames(RNASeq))

# log transformation
# log transformation with handling negative values
# log transformation with handling negative values
RNASeq <- log2(pmax(RNASeq, 0) + 0.1)
RNASeq <- t(RNASeq)
rownames(RNASeq) <- sub("\\.[0-9]{2}$", "", rownames(RNASeq))
SDs <- apply(RNASeq,2,sd)
orderSD <- order(SDs, decreasing = T)[1:2000]
RNASeq <- RNASeq[,orderSD]

dim(RNASeq)

In [ ]:
head(colnames(RNASeq))

In [ ]:
head(RNASeq,n=15)

In [ ]:
length(unique(colnames(RNASeq)))# there are duplicated variables

In [ ]:
head(RNASeq,n=15)

In [ ]:
copyNumber <- listAssays[[1]]
colnames(copyNumber) <- gsub("(\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[0-9]+)$", "", colnames(copyNumber))
colnames(copyNumber) <- gsub("\\-|\\_", ".", colnames(copyNumber))
rownames(copyNumber) <- gene_symbol_df$Gene.Symbol
# select patients with only ductal or lobular carcinoma
 copyNumber <- copyNumber[,intersect(colnames(copyNumber), rownames(dfcolData))]
 dim(copyNumber)

In [ ]:
head(copyNumber)

In [ ]:
length(unique(colnames(copyNumber))) ## there are unique variables

In [ ]:
copyNumber <- t(copyNumber)

In [ ]:
head(colnames(copyNumber))

In [ ]:
head(copyNumber)

In [ ]:
dim(copyNumber)

In [ ]:
## mutation
Mutation <- listAssays[[2]]
colnames(Mutation) <- gsub("(\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[0-9]+)$", "", colnames(Mutation))

# reshape the sparse matrix and set binary values for mutations
Mutationdf <- as.data.frame(Mutation)
tmp <- data.frame(GeneSymbol = rep(NA, nrow(Mutationdf)))
for(i in 1:ncol(Mutationdf)){
  tmp$GeneSymbol <- coalesce(Mutationdf[,i], tmp$GeneSymbol)
  Mutationdf$GeneSymbol <- tmp$GeneSymbol
}

sum(is.na(Mutationdf$GeneSymbol))# no missing value
dim(Mutation)

In [ ]:
head(Mutation)

In [ ]:
tmp <- data.frame(GeneSymbol = unique(Mutationdf$GeneSymbol))# select unique gene symbols
sum(is.na(tmp$GeneSymbol))# no NA


In [ ]:
cl <- makeCluster(3)
registerDoParallel(cl)


In [ ]:
dim(Mutationdf)

In [ ]:
tmp_data <- foreach(i = c(1:232), j = c(1:231))%do%{
  tmp[,paste0("V", i)] <- ifelse(tmp$GeneSymbol%in%Mutationdf[,j], 1, 0)
}

In [ ]:
tmp_df <- data.frame(matrix(unlist(tmp_data), ncol = length(tmp_data)))# convert to data frame
colnames(tmp_df) <- colnames(Mutationdf[,1:231])
rownames(tmp_df) <- tmp$GeneSymbol
Mutationdf <- tmp_df
colnames(Mutationdf) <- gsub("\\-|\\_", ".", colnames(Mutationdf))

# select patients with only ductal or lobular carcinoma
Mutationdf <- Mutationdf[,intersect(colnames(Mutationdf), rownames(dfcolData))]
dim(Mutationdf)

In [ ]:
length(unique(colnames(Mutationdf)))# unique variables

In [ ]:
rm(tmp, tmp_data, tmp_df)
Mutation <- t(as.matrix(Mutationdf))

# filter out genes with too few mutations (less than 5%)
mut.rate <- apply(Mutation, 2, mean)
Mutation <- Mutation[,which(mut.rate>0.02)]
dim(Mutation)

In [ ]:
head(Mutation)

In [ ]:
## PP
RPPA <- listAssays[[3]]
dfRPPA <- data.frame(matrix(nrow = 226))

In [ ]:
head(RPPA)

In [ ]:
# Assuming RPPA is a matrix
dfRPPA <- matrix(NA, nrow = nrow(RPPA), ncol = 0)  # Initialize an empty matrix

for (i in 1:ncol(RPPA)) {
  # Check if the number of rows matches
  if (nrow(dfRPPA) == nrow(RPPA[, i, drop = FALSE])) {
    dfRPPA <- cbind(dfRPPA, RPPA[, i, drop = FALSE])
  } else {
    warning("Number of rows mismatch in iteration ", i)
    # Handle the mismatch (e.g., skip or stop the loop)
    # You might need to adjust your logic based on your requirements
  }
}

# Continue with the rest of your code
dfRPPA <- dfRPPA[, -1]
rownames(dfRPPA) <- rownames(RPPA)
colnames(dfRPPA) <- gsub("(\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[[:alnum:]]+\\-[0-9]+)$", "", colnames(dfRPPA))
rownames(dfRPPA) <- gsub("\\-|\\_", ".", rownames(dfRPPA))

length(unique(colnames(dfRPPA)))  # Check for duplicated variables


In [ ]:
dim(dfRPPA)
head(dfRPPA)

In [ ]:
dfRPPA <- dfRPPA[,unique(colnames(dfRPPA))]
colnames(dfRPPA) <- gsub("\\-|\\_", ".", colnames(dfRPPA))

# select patients with only ductal or lobular carcinoma
 dfRPPA <- dfRPPA[,intersect(colnames(dfRPPA), rownames(dfcolData))]
mRPPA <- t(as.matrix(dfRPPA))
dim(mRPPA)

In [ ]:
head(mRPPA)

In [ ]:
sum(is.na(mRPPA))

In [ ]:
# Identify variables to be removed with more than 50% missing values
toBeRemoved <- which(data.frame(map(as.data.frame(mRPPA), ~sum(is.na(.)))) > nrow(mRPPA)/2)

# Remove variables with more than 50% missing values
mRPPA <- mRPPA[, -c(toBeRemoved)]

# Check for missing values and impute the rest
missing_values <- any(is.na(mRPPA))
if (missing_values) {
  set.seed(4578)
  registerDoParallel(cores = 1)
  impute_forest <- missForest(data.frame(mRPPA), maxiter = 10, ntree = 100, parallelize = "no")
  stopImplicitCluster()
  mRPPA <- impute_forest$ximp
}

# Continue with the rest of your analysis using mRPPA


In [ ]:
impute_forest$OOBerror

In [ ]:
mRPPA <- as.matrix(impute_forest$ximp)
sum(apply(mRPPA, 1, sum))

In [ ]:
# Assuming mRPPA is your dataframe
# Add prefix "rs_" to every column name
colnames(mRPPA) <- paste0("pp_", colnames(mRPPA))

# Display the modified dataframe
head(mRPPA)
dim(mRPPA)

In [ ]:
colnames(Mutation) <- paste0("mu_", colnames(Mutation))

head(Mutation)
dim(Mutation)

In [ ]:
colnames(RNASeq) <- paste0("rs_", colnames(RNASeq))

head(RNASeq)
dim(RNASeq)

In [ ]:
colnames(copyNumber) <- paste0("cn_", colnames(copyNumber))

head(copyNumber)
dim(copyNumber)

In [ ]:
#### list of omics with same patients
dataList <- list(CN =copyNumber,MU= Mutation,RS= RNASeq,PP= mRPPA)
intersectID <- Reduce(intersect, lapply(dataList, rownames))
dataList <- lapply(dataList, function(x) x[intersectID,]) # data is selected for 705 patients
finalData <- cbind(dataList$CN, dataList$MU, dataList$RS, dataList$PP)
## filter colData by shared rows
dfcolData <- dfcolData[intersect(rownames(dfcolData), rownames(dataList$CN)),]

In [ ]:
head(dfcolData)

In [ ]:
write.csv(dfcolData, "metastasisstatusb1.csv", row.names = FALSE)

In [ ]:
dim(finalData)

In [ ]:
# Check if cn_RN7SL105P is a column name
if ("cn_RN7SL105P" %in% colnames(finalData)) {
  print("Column cn_RN7SL105P exists in finalData.")
} else {
  print("Column cn_RN7SL105P does not exist in finalData.")
}

In [ ]:
# Assuming dataList contains CN, MU, RS, and PP data
set.seed(123)  # Setting seed for reproducibility

# Randomly shuffle columns for each omics type
shuffledCN <- dataList$CN[, sample(ncol(dataList$CN))]
shuffledMU <- dataList$MU[, sample(ncol(dataList$MU))]
shuffledRS <- dataList$RS[, sample(ncol(dataList$RS))]
shuffledPP <- dataList$PP[, sample(ncol(dataList$PP))]

# Define the number of columns for each omics type
numColsCN <- 860
numColsMU <- 249
numColsRS <- 604
numColsPP <- 218

# Select the specified number of columns for each omics type
finalDataset <- cbind(shuffledCN[, 1:numColsCN], shuffledMU[, 1:numColsMU], shuffledRS[, 1:numColsRS], shuffledPP[, 1:numColsPP])

In [ ]:
# Load data frames
batchone <- read.csv("batchone.csv")

# Get column names
finalData_cols <- colnames(finalData)
batchone_cols <- colnames(batchone)

# Identify columns to delete
columns_to_delete <- setdiff(finalData_cols, batchone_cols)

# Delete columns from batchone.csv
batchone <- batchone[, !(names(batchone) %in% columns_to_delete)]

In [ ]:
dim(batchone)

In [ ]:
# Load data frames
batchone <- read.csv("batchone.csv")

# Get column names
finalData_cols <- colnames(finalData)
batchone_cols <- colnames(batchone)

# Identify columns to keep
columns_to_keep <- intersect(finalData_cols, batchone_cols)

# Filter finalData to include only matching columns
finalData_filtered <- finalData[, columns_to_keep]

# Filter batchone to include only matching columns
batchone_filtered <- batchone[, columns_to_keep]

# Now batchone_filtered and finalData_filtered both contain only matching columns


In [ ]:
dim(finalData_filtered)
dim(batchone_filtered)

In [ ]:
head(finalData_filtered, n = 100)

In [ ]:
head(batchone_filtered, n = 100)

In [ ]:
# Save filtered datasets as CSV files
write.csv(finalData_filtered, "batchtwo_filtered.csv", row.names = FALSE)


In [ ]:
write.csv(batchone_filtered, "batchone_filtered.csv", row.names = FALSE)

In [ ]:
# Load the original data
finalData <- read.csv("NSCLCMultiOmicsDataProcessed.csv")  # Adjust the file path

# Get column names
column_names <- colnames(finalData)

# Define the order of prefixes
column_order <- c("rs_", "cn_", "mu_", "pp_")

# Reorder columns in the finalData data frame
finalData_reordered <- finalData %>%
  select(starts_with(column_order))

In [ ]:
write.csv(finalData_reordered, "NSCLCMultiOmicsDataProcessed_reordered.csv", row.names = FALSE)